# Notebook 3: Model Evaluation & Comparison

Computes mAP, IoU, Precision-Recall curves, and error analysis for all three models.
Generates the final comparison table required for the report.


In [ ]:
import sys, time
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torchvision.transforms.functional as TF
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

PROJECT_ROOT = Path('..').resolve()
DATA_DIR     = PROJECT_ROOT / 'data'
RUNS_DIR     = PROJECT_ROOT / 'runs'
sys.path.insert(0, str(PROJECT_ROOT))

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES    = ['cracks', 'spalling', 'corrosion', 'potholes', 'paint_degradation']
NUM_CLASSES = len(CLASSES) + 1
CONF_THRESH = 0.25
IOU_THRESH  = 0.50
IMG_SIZE    = 640

## 1. Load Models


In [ ]:
from ultralytics import YOLO
from models.resnet_detector import load_checkpoint as load_resnet
from models.efficientnet_detector import load_checkpoint as load_effnet

yolo_weights   = RUNS_DIR / 'yolo'        / 'exp1'    / 'weights' / 'best.pt'
resnet_weights = RUNS_DIR / 'resnet50'    / 'best.pth'
eff_weights    = RUNS_DIR / 'efficientnet'/ 'best.pth'

models = {}
if yolo_weights.exists():
    models['YOLOv11n'] = YOLO(str(yolo_weights))
    print('Loaded YOLOv11n')
if resnet_weights.exists():
    models['ResNet50'] = load_resnet(str(resnet_weights), DEVICE)
    print('Loaded ResNet50')
if eff_weights.exists():
    models['EfficientNetB0'] = load_effnet(str(eff_weights), DEVICE)
    print('Loaded EfficientNetB0')

print(f'Models loaded: {list(models.keys())}')

## 2. Evaluation Functions


In [ ]:
from torch.utils.data import DataLoader, Dataset


class TestDataset(Dataset):
    def __init__(self, split: str = 'test'):
        self.img_dir = DATA_DIR / 'images' / split
        self.lbl_dir = DATA_DIR / 'labels' / split
        self.images  = sorted(self.img_dir.glob('*.jpg'))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        lbl_path = self.lbl_dir / (img_path.stem + '.txt')
        img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        tensor = TF.to_tensor(img)
        boxes, labels = [], []
        if lbl_path.exists():
            for line in lbl_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
                x1 = max(0.0, (cx - bw / 2) * IMG_SIZE)
                y1 = max(0.0, (cy - bh / 2) * IMG_SIZE)
                x2 = min(float(IMG_SIZE), (cx + bw / 2) * IMG_SIZE)
                y2 = min(float(IMG_SIZE), (cy + bh / 2) * IMG_SIZE)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(cls + 1)
        if boxes:
            boxes_t  = torch.tensor(boxes,  dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes_t  = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros(0,      dtype=torch.int64)
        return tensor, {'boxes': boxes_t, 'labels': labels_t, 'image_id': torch.tensor([idx])}


def collate_fn(batch):
    return tuple(zip(*batch))


test_ds = TestDataset('test')
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)
print(f'Test images: {len(test_ds)}')

In [ ]:
@torch.no_grad()
def evaluate_torchvision(model, loader, device, conf=CONF_THRESH):
    model.eval()
    metric = MeanAveragePrecision(iou_type='bbox', class_metrics=True)
    times = []
    for images, targets in tqdm(loader):
        imgs = [img.to(device) for img in images]
        t0 = time.time()
        outputs = model(imgs)
        times.append((time.time() - t0) / len(imgs) * 1000)  # ms/image
        preds = []
        for o in outputs:
            mask = o['scores'] >= conf
            preds.append({'boxes':  o['boxes'][mask].cpu(),
                          'scores': o['scores'][mask].cpu(),
                          'labels': o['labels'][mask].cpu()})
        gts = [{'boxes':  t['boxes'].cpu(),
                'labels': t['labels'].cpu()} for t in targets]
        metric.update(preds, gts)
    result = metric.compute()
    result['inference_ms'] = np.mean(times)
    return result


def evaluate_yolo(model, data_yaml, split='test'):
    return model.val(data=data_yaml, split=split)


def measure_yolo_inference_speed(model, n_images: int = 50):
    imgs = list((DATA_DIR / 'images' / 'test').glob('*.jpg'))[:n_images]
    times = []
    for p in imgs:
        t0 = time.time()
        model.predict(str(p), verbose=False)
        times.append((time.time() - t0) * 1000)
    return np.mean(times)

## 3. Run Evaluation on Test Set


In [ ]:
DATA_YAML = str(DATA_DIR / 'data.yaml')
results = {}

if 'YOLOv11n' in models:
    print('\n--- Evaluating YOLOv11n ---')
    yolo_val = evaluate_yolo(models['YOLOv11n'], DATA_YAML, split='test')
    yolo_speed = measure_yolo_inference_speed(models['YOLOv11n'])
    results['YOLOv11n'] = {
        'mAP@0.5':     round(float(yolo_val.box.map50),  4),
        'mAP@0.5:0.95':round(float(yolo_val.box.map),    4),
        'Precision':   round(float(yolo_val.box.mp),      4),
        'Recall':      round(float(yolo_val.box.mr),      4),
        'Speed (ms)':  round(yolo_speed, 2),
    }

if 'ResNet50' in models:
    print('\n--- Evaluating ResNet-50 ---')
    r = evaluate_torchvision(models['ResNet50'], test_loader, DEVICE)
    results['ResNet50'] = {
        'mAP@0.5':     round(float(r['map_50']),       4),
        'mAP@0.5:0.95':round(float(r['map']),          4),
        'Precision':   None,
        'Recall':      None,
        'Speed (ms)':  round(float(r['inference_ms']), 2),
    }

if 'EfficientNetB0' in models:
    print('\n--- Evaluating EfficientNet-B0 ---')
    r = evaluate_torchvision(models['EfficientNetB0'], test_loader, DEVICE)
    results['EfficientNetB0'] = {
        'mAP@0.5':     round(float(r['map_50']),       4),
        'mAP@0.5:0.95':round(float(r['map']),          4),
        'Precision':   None,
        'Recall':      None,
        'Speed (ms)':  round(float(r['inference_ms']), 2),
    }

## 4. Model Comparison Table


In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df.index.name = 'Model'
print('\n=== Model Comparison Table ===')
print(comparison_df.to_string())

comparison_df.to_csv(RUNS_DIR / 'model_comparison.csv')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metrics_to_plot = ['mAP@0.5', 'mAP@0.5:0.95']
colors = ['#3498db', '#e74c3c', '#2ecc71']
for i, metric in enumerate(metrics_to_plot):
    vals = [results[m][metric] for m in results if results[m][metric] is not None]
    names = [m for m in results if results[m][metric] is not None]
    axes[i].bar(names, vals, color=colors[:len(names)], edgecolor='black')
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].set_ylabel('Score')
    for j, v in enumerate(vals):
        axes[i].text(j, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)
plt.suptitle('Model Comparison', fontsize=14)
plt.tight_layout()
plt.savefig(str(RUNS_DIR / 'model_comparison.png'), dpi=150)
plt.show()

## 5. Per-Class AP (YOLO)


In [ ]:
if 'YOLOv11n' in models:
    per_class = dict(zip(CLASSES, yolo_val.box.maps))
    fig, ax = plt.subplots(figsize=(8, 4))
    colors_bar = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
    ax.bar(list(per_class.keys()), list(per_class.values()), color=colors_bar, edgecolor='black')
    ax.set_title('YOLOv11n — Per-Class AP@0.5')
    ax.set_ylabel('AP@0.5')
    ax.set_ylim(0, 1)
    for i, (k, v) in enumerate(per_class.items()):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(str(RUNS_DIR / 'per_class_ap_yolo.png'), dpi=150)
    plt.show()

## 6. Severity Distribution on Test Set


In [ ]:
def severity_label(bbox_w_norm: float, bbox_h_norm: float) -> str:
    """Returns Low/Medium/High based on fractional bbox area."""
    area_pct = bbox_w_norm * bbox_h_norm * 100
    if area_pct < 5:
        return 'Low'
    elif area_pct <= 20:
        return 'Medium'
    return 'High'


severity_counts = {'Low': 0, 'Medium': 0, 'High': 0}
for lbl_path in (DATA_DIR / 'labels' / 'test').glob('*.txt'):
    for line in lbl_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) == 5:
            bw, bh = float(parts[3]), float(parts[4])
            severity_counts[severity_label(bw, bh)] += 1

print('Ground-truth severity distribution in test set:')
for sev, cnt in severity_counts.items():
    print(f'  {sev}: {cnt}')

fig, ax = plt.subplots(figsize=(5, 4))
ax.pie(severity_counts.values(), labels=severity_counts.keys(),
       autopct='%1.1f%%', colors=['#2ecc71', '#f39c12', '#e74c3c'])
ax.set_title('Ground-Truth Severity Distribution (Test Set)')
plt.tight_layout()
plt.savefig(str(RUNS_DIR / 'severity_distribution.png'), dpi=150)
plt.show()

## 7. Qualitative Prediction Visualisation (YOLO)


In [ ]:
if 'YOLOv11n' in models:
    CLASS_COLORS = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']
    test_imgs = sorted((DATA_DIR / 'images' / 'test').glob('*.jpg'))[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    for i, img_path in enumerate(test_imgs):
        result = models['YOLOv11n'].predict(str(img_path), conf=CONF_THRESH, verbose=False)[0]
        img = Image.open(img_path).convert('RGB')
        axes[i].imshow(img)
        w, h = img.size
        for box in result.boxes:
            cls   = int(box.cls.item())
            conf  = float(box.conf.item())
            x1,y1,x2,y2 = box.xyxy[0].tolist()
            bw_n = (x2 - x1) / w
            bh_n = (y2 - y1) / h
            sev  = severity_label(bw_n, bh_n)
            label = f'{CLASSES[cls]} {conf:.2f} [{sev}]'
            rect  = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                       linewidth=2,
                                       edgecolor=CLASS_COLORS[cls],
                                       facecolor='none')
            axes[i].add_patch(rect)
            axes[i].text(x1, y1-5, label, color=CLASS_COLORS[cls],
                         fontsize=7, fontweight='bold')
        axes[i].axis('off')
        axes[i].set_title(img_path.name, fontsize=8)

    plt.suptitle('YOLOv11n Predictions with Severity', fontsize=13)
    plt.tight_layout()
    plt.savefig(str(RUNS_DIR / 'prediction_samples.png'), dpi=120)
    plt.show()